# DateTime Features: Extracting Temporal Patterns

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadtalhaishtiaq/ai-orchestrator/blob/main/03-feature-engineering/06_datetime_features.ipynb)

## Objectives
- Extract components from datetime (year, month, day, etc.)
- Create temporal features (day of week, quarter, season)
- Calculate time differences and durations
- Handle cyclical datetime features
- Capture business logic (holidays, weekends)

print("""📅 DATETIME FEATURES:

1. DIRECT EXTRACTION:
   • Year, Month, Day
   • Hour, Minute, Second
   • Week number, Quarter

2. DERIVED TEMPORAL:
   • Day of week (Monday=0 to Sunday=6)
   • Is weekend (boolean)
   • Is month end/start
   • Is quarter end
   • Days since epoch

3. SEASONAL:
   • Season (Spring, Summer, Fall, Winter)
   • Quarter
   • Is holiday
   • School term

4. CYCLIC:
   • sin/cos encoding for cyclical features
   • Day of year (0-365, but 365 ≈ 0)
   • Month (12 ≈ 0)

5. DIFFERENCES:
   • Time since event (days/hours)
   • Duration between events
   • Time to next event""")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

np.random.seed(42)
sns.set_theme()

print("✅ Libraries loaded")

# Create date range
date_range = pd.date_range(start='2022-01-01', end='2024-12-31', freq='D')
np.random.seed(42)
sample_dates = np.random.choice(date_range, size=500, replace=False)
sample_dates = sorted(sample_dates)

df = pd.DataFrame({
    'date': sample_dates,
    'purchase_amount': np.random.exponential(50, 500) + 10,
    'user_signup_date': pd.date_range('2020-01-01', periods=500, freq='D').shift(freq=np.random.randint(-500, 500, 500))
})

print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head(10))
print(f"\nData types:")
print(df.dtypes)

# Extract basic components
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['quarter'] = df['date'].dt.quarter
df['week'] = df['date'].dt.isocalendar().week
df['day_of_year'] = df['date'].dt.day_of_year

print("📊 Basic DateTime Components:")
print(df[['date', 'year', 'month', 'day', 'quarter', 'week', 'day_of_year']].head(10))

# Statistics
print(f"\nYear range: {df['year'].min()} to {df['year'].max()}")
print(f"Month distribution:\n{df['month'].value_counts().sort_index()}")

# Day of week (0=Monday, 6=Sunday)
df['day_of_week'] = df['date'].dt.day_of_week
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

# Day names
day_names = {0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday',
            4: 'Friday', 5: 'Saturday', 6: 'Sunday'}
df['day_name'] = df['day_of_week'].map(day_names)

# Month-related
df['is_month_start'] = df['date'].dt.is_month_start.astype(int)
df['is_month_end'] = df['date'].dt.is_month_end.astype(int)
df['is_quarter_start'] = df['date'].dt.is_quarter_start.astype(int)
df['is_quarter_end'] = df['date'].dt.is_quarter_end.astype(int)

print("📊 Temporal Pattern Features:")
print(df[['date', 'day_of_week', 'day_name', 'is_weekend', 'is_month_end']].head(15))

print(f"\nWeekend vs Weekday purchases:")
print(f"  Weekend: {df[df['is_weekend']==1]['purchase_amount'].mean():.2f}")
print(f"  Weekday: {df[df['is_weekend']==0]['purchase_amount'].mean():.2f}")

# Define seasons by month
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'

df['season'] = df['month'].apply(get_season)

# Simple holidays (example)
def is_holiday(date):
    holidays = [
        (12, 25),  # Christmas
        (1, 1),    # New Year
        (7, 4),    # Independence Day
        (11, 25),  # Thanksgiving (approx)
    ]
    return (date.month, date.day) in holidays

df['is_holiday'] = df['date'].apply(is_holiday).astype(int)

print("📊 Seasonal Features:")
print(f"\nSeason distribution:")
print(df['season'].value_counts())

print(f"\nAverage purchase by season:")
print(df.groupby('season')['purchase_amount'].mean())

print(f"\nHoliday transactions: {df['is_holiday'].sum()}")

# Important: Day of year is cyclical (365 and 0 are neighbors)
# Need to use sin/cos encoding to preserve this

# Method 1: Without sin/cos (WRONG - loses cyclical nature)
df['day_of_year_raw'] = df['date'].dt.day_of_year

# Method 2: With sin/cos (CORRECT - preserves cyclical)
df['day_of_year_sin'] = np.sin(2 * np.pi * df['day_of_year_raw'] / 365)
df['day_of_year_cos'] = np.cos(2 * np.pi * df['day_of_year_raw'] / 365)

# Same for month
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

print("📊 Cyclical Features (Sin/Cos Encoding):")
print(df[['day_of_year_raw', 'day_of_year_sin', 'day_of_year_cos', 'month', 'month_sin', 'month_cos']].head(10))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw day of year (linear - problematic)
axes[0].scatter(df['day_of_year_raw'], df['purchase_amount'], alpha=0.5, s=20)
axes[0].set_xlabel('Day of Year')
axes[0].set_ylabel('Purchase Amount')
axes[0].set_title('Raw Day of Year (Linear - Loses Cyclicity)')
axes[0].axvline(x=365, color='red', linestyle='--', label='End of year')
axes[0].axvline(x=1, color='green', linestyle='--', label='Start of year')
axes[0].legend()

# Sin/Cos (circular)
axes[1].scatter(df['day_of_year_sin'], df['day_of_year_cos'], c=df['purchase_amount'], 
               cmap='viridis', alpha=0.6, s=30)
axes[1].set_xlabel('sin(day_of_year)')
axes[1].set_ylabel('cos(day_of_year)')
axes[1].set_title('Sin/Cos Encoding (Circular - Preserves Cyclicity)')
axes[1].set_aspect('equal')
circle = plt.Circle((0, 0), 1, color='black', fill=False, linestyle='--', alpha=0.3)
axes[1].add_patch(circle)
plt.colorbar(axes[1].collections[0], ax=axes[1], label='Purchase Amount')

plt.tight_layout()
plt.show()

print("\n💡 Notice: Sin/Cos keeps Dec 31 (day ~365) close to Jan 1 (day ~1)")

# Days since signup
df['days_since_signup'] = (df['date'] - df['user_signup_date']).dt.days
df['days_since_signup'] = df['days_since_signup'].clip(lower=0)  # Can't be negative

# Days since epoch (reference time)
epoch = pd.Timestamp('2020-01-01')
df['days_since_epoch'] = (df['date'] - epoch).dt.days

print("📊 Time Difference Features:")
print(df[['date', 'user_signup_date', 'days_since_signup', 'days_since_epoch']].head(15))

print(f"\nDays since signup statistics:")
print(df['days_since_signup'].describe())

# Sort by date first
df_sorted = df.sort_values('date').reset_index(drop=True)

# Lagged features
df_sorted['purchase_prev_day'] = df_sorted['purchase_amount'].shift(1)
df_sorted['purchase_prev_week'] = df_sorted['purchase_amount'].shift(7)

# Rolling statistics
df_sorted['purchase_rolling_mean_7d'] = df_sorted['purchase_amount'].rolling(window=7).mean()
df_sorted['purchase_rolling_std_7d'] = df_sorted['purchase_amount'].rolling(window=7).std()

print("📊 Lagged & Rolling Features:")
print(df_sorted[['date', 'purchase_amount', 'purchase_prev_day', 'purchase_rolling_mean_7d']].head(20))

# Visualization
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_sorted['date'], df_sorted['purchase_amount'], alpha=0.5, label='Daily', linewidth=1)
ax.plot(df_sorted['date'], df_sorted['purchase_rolling_mean_7d'], label='7-Day Mean', linewidth=2, color='red')
ax.fill_between(df_sorted['date'], 
                 df_sorted['purchase_rolling_mean_7d'] - df_sorted['purchase_rolling_std_7d'],
                 df_sorted['purchase_rolling_mean_7d'] + df_sorted['purchase_rolling_std_7d'],
                 alpha=0.2, label='±1 Std Dev')
ax.set_xlabel('Date')
ax.set_ylabel('Purchase Amount')
ax.set_title('Rolling Statistics: Capturing Temporal Patterns')
ax.legend()
plt.tight_layout()
plt.show()

def extract_datetime_features(df, date_col, date_format=None):
    """Extract comprehensive datetime features"""
    df = df.copy()
    
    # Ensure datetime format
    if df[date_col].dtype != 'datetime64[ns]':
        df[date_col] = pd.to_datetime(df[date_col], format=date_format)
    
    dt = df[date_col]
    
    # Basic components
    df[f'{date_col}_year'] = dt.dt.year
    df[f'{date_col}_month'] = dt.dt.month
    df[f'{date_col}_day'] = dt.dt.day
    df[f'{date_col}_quarter'] = dt.dt.quarter
    df[f'{date_col}_week'] = dt.dt.isocalendar().week
,
,
,
,
5
6
,
,
,
,
,
365
,
365
,
12
,
12
,
,
,
,
,
,
,
✅ Features before: {len(df.columns)}")
print(f"✅ Features after: {len(df_features.columns)}")
print(f"✅ New features added: {len(df_features.columns) - len(df.columns)}")

print(f"\nNew datetime features:")
datetime_cols = [col for col in df_features.columns if 'date_' in col]
print(datetime_cols)

print("""\n📚 KEY TAKEAWAYS:

DateTime Feature Types:
✓ Basic: Year, month, day, hour
✓ Temporal: Day of week, week number, quarter
✓ Seasonal: Season, quarter, holidays
✓ Cyclical: Sin/Cos encoding for circular features
✓ Differences: Days since event, durations
✓ Lagged: Previous values, rolling statistics

☑️ CRITICAL: Handle Cyclicity
   • Wrong: Use day_of_year (1-365) directly
   • Right: Use sin/cos encoding
   • Reason: Dec 31 (365) and Jan 1 (1) are neighbors

✅ Best Practices:
   1. Extract for specific business logic
   2. Handle timezones consistently
   3. Account for missing/leap years
   4. Use proper cyclical encoding
   5. Validate on domain knowledge

📊 Common Uses:
   • Sales forecasting (seasonal patterns)
   • Customer behavior (weekend vs weekday)
   • Demand prediction (time-based features)
   • Anomaly detection (unusual times)

Next: Text feature engineering!""")